# Notebook 23 — Full Paper Generator

**Purpose:** generate a complete paper scaffold from Notebooks 16–22: LaTeX sections, figure registry, tables, bibliography stub, Makefile, and a template-consistent outputs zip.

**Claim guardrail:** empirical computational evidence only; no asymptotic theorem or prime conjecture proof claim.


In [ ]:
from __future__ import annotations
import os, re, json, math, shutil, zipfile, subprocess
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

NOTEBOOK_NUM = 23
NOTEBOOK_ID = "23_full_paper_generator"
NOTEBOOK_TITLE = "Higher-Order Residue Memory in Prime Gap Transitions"
AUTHOR = "Dan Hawkley"
REPO_URL = "github.com/thinkthoughts/prime-numbers-lab"
SEED = 9423
rng = np.random.default_rng(SEED)

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / f"{NOTEBOOK_ID}_outputs"
PAPER_DIR = OUTPUT_DIR / "paper"
SECTIONS_DIR = PAPER_DIR / "sections"
FIGURES_DIR = PAPER_DIR / "figures"
TABLES_DIR = PAPER_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "data"
DOCS_DIR = OUTPUT_DIR / "docs"
TEX_DIR = OUTPUT_DIR / "tex"
for d in [OUTPUT_DIR, PAPER_DIR, SECTIONS_DIR, FIGURES_DIR, TABLES_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_ZIP = ROOT / f"{NOTEBOOK_ID}_outputs.zip"
MANIFEST_CSV = OUTPUT_DIR / f"{NOTEBOOK_NUM}_outputs_manifest.csv"
SUMMARY_CSV = OUTPUT_DIR / f"{NOTEBOOK_NUM}_interpretation_summary.csv"
print("NOTEBOOK_ID:", NOTEBOOK_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PAPER_DIR:", PAPER_DIR)


## 1. Discover prior artifacts

In [ ]:
def safe_slug(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", s.strip())
    return re.sub(r"_+", "_", s).strip("_").lower()

SEARCH_ROOTS = [ROOT / "figures", ROOT / "outputs", ROOT / "data", ROOT / "docs", ROOT / "tex", ROOT]
for n in range(16, 23):
    SEARCH_ROOTS.extend([ROOT / f"{n}_outputs", ROOT / f"{n:02d}_outputs", ROOT / f"notebook_{n}_outputs"])
SEARCH_ROOTS.extend(sorted(ROOT.glob("*_outputs")))
SEARCH_ROOTS = [p for p in dict.fromkeys(SEARCH_ROOTS) if p.exists()]

prior_manifests = []
for base in SEARCH_ROOTS:
    for pat in ["*manifest*.csv", "*manifest*.json"]:
        for f in base.rglob(pat):
            if f.is_file() and str(f) not in {str(MANIFEST_CSV)}:
                prior_manifests.append(f)
prior_manifests = sorted(set(prior_manifests), key=lambda p: str(p))
print(f"Found {len(prior_manifests)} prior manifest-like files")
for p in prior_manifests[:20]:
    print(" -", p.relative_to(ROOT) if p.is_relative_to(ROOT) else p)


## 2. Select and copy paper figures

In [ ]:
preferred_keywords = [
    "16_transition_operator_heatmap", "16_windowed_transition_entropy", "16_mixing_distance_vs_steps",
    "17_real_vs_shuffle_two_step_l2", "17_empirical_two_step_operator_heatmap", "17_two_step_delta_operator",
    "18_corrected_two_step_operator", "18_predictive_error_versus_memory_rank", "18_windowed_l2_residual_before_after_rank4_correction",
    "20_real_vs_controls_singular_spectrum", "20_real_vs_controls_delta_heatmaps", "20_mutual_information_control_comparison", "20_control_residual_l2_comparison",
    "21_statistical_validation_pvalue_heatmap", "21_publication_summary_real_metric_over_null_mean", "21_singular_spectrum_real_vs_null_confidence_bands", "21_bootstrap_confidence_intervals", "21_rank_requirement_stability",
]
all_pngs = sorted({f for base in SEARCH_ROOTS for f in base.rglob("*.png") if f.is_file()}, key=lambda p: str(p))

def score_figure(p: Path) -> int:
    name = p.stem
    score = 0
    for i, kw in enumerate(preferred_keywords):
        if kw in name:
            score += 1000 - i
    if re.match(r"^(16|17|18|20|21)_", name): score += 50
    if any(w in name for w in ["heatmap", "spectrum", "validation", "control", "operator", "rank"]): score += 20
    if "manifest" in name.lower(): score -= 100
    return score
selected=[]; seen=set()
for p in sorted(all_pngs, key=lambda p: score_figure(p), reverse=True):
    if score_figure(p) <= 0: continue
    if p.stem in seen: continue
    selected.append(p); seen.add(p.stem)
    if len(selected) >= 12: break
figure_rows=[]
for idx, src in enumerate(selected, start=1):
    dest_name = f"fig{idx:02d}_{safe_slug(src.stem)}.png"
    dest = FIGURES_DIR / dest_name
    shutil.copy2(src, dest)
    figure_rows.append({"figure_id": f"fig:{idx:02d}", "source_path": str(src.relative_to(ROOT)) if src.is_relative_to(ROOT) else str(src), "paper_path": f"figures/{dest_name}", "filename": dest_name, "caption": ""})
print(f"Selected and copied {len(figure_rows)} figures")
for row in figure_rows: print(row["figure_id"], row["paper_path"], "<-", row["source_path"])


## 3. Caption registry

In [ ]:
def caption_for(filename: str) -> str:
    n = filename.lower()
    if "transition_operator" in n and "heatmap" in n: return "Empirical residue-class transition operator for consecutive primes modulo 30."
    if "entropy" in n: return "Windowed entropy-rate diagnostics for residue-transition structure across scale."
    if "mixing" in n: return "Mixing-distance diagnostics for fitted transition dynamics."
    if "real_vs_shuffle" in n or ("shuffle" in n and "l2" in n): return "Real two-step memory compared against shuffle controls."
    if "two_step_delta" in n or "delta_operator" in n or "delta_heatmap" in n: return "Two-step residual operator Delta = P^(2) - P^2, highlighting deviations from a first-order Markov baseline."
    if "corrected_two_step" in n or ("rank" in n and "correction" in n): return "Low-rank correction to the empirical two-step operator using leading residual modes."
    if "predictive_error" in n: return "Predictive error as a function of retained memory rank."
    if "singular_spectrum" in n or "spectrum" in n: return "Singular spectrum of the residual operator compared with null or control ensembles."
    if "mutual_information" in n or "mi_" in n: return "Mutual-information excess used to quantify dependency beyond shuffled baselines."
    if "control_residual" in n or "controls" in n: return "Control comparison for residual operator strength across real and synthetic sequences."
    if "validation" in n or "pvalue" in n or "p_value" in n: return "Statistical validation summary against null and resampling ensembles."
    if "bootstrap" in n: return "Bootstrap confidence intervals for real-sequence statistics."
    if "rank_requirement" in n: return "Rank requirement stability for explaining residual spectral energy."
    return "Paper-ready diagnostic figure generated from the prime-residue transition pipeline."
for row in figure_rows: row["caption"] = caption_for(row["filename"])
fig_registry = pd.DataFrame(figure_rows)
fig_registry.to_csv(TABLES_DIR / "paper_figure_registry.csv", index=False)
fig_registry


## 4. Metadata and LaTeX sections

In [ ]:
def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.strip() + "\n", encoding="utf-8")
    return path
paper_meta = {"title": NOTEBOOK_TITLE, "author": AUTHOR, "repo_url": REPO_URL, "notebook_id": NOTEBOOK_ID, "generated_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z", "claim_guardrail": "Empirical computational evidence only; no asymptotic theorem or RH-related proof claim."}
write_text(DATA_DIR / "paper_metadata.json", json.dumps(paper_meta, indent=2))
sections = {'00_abstract.tex': 'We study residue-class transitions of consecutive primes modulo 30 as a finite computational dynamical system. Using the first-order transition operator P as a Markov baseline, we compare the empirical two-step operator P^{(2)} against P^2 and analyze the residual operator Delta = P^{(2)} - P^2. Across the computed range, the residual exhibits low-rank structure, stable singular modes, and measurable prediction improvements under low-rank correction. Synthetic controls, including iid shuffles, Markov-generated sequences, block shuffles, balanced shuffles, and gap-based shuffles, show that the observed structure is not explained by marginal residue frequencies or first-order transition statistics alone. Bootstrap and null validation indicate statistically meaningful separation from structure-destroying controls while remaining consistent with local block-preserving resampling. These results provide empirical computational evidence for higher-order residue memory in finite prime-gap transition data, without claiming an asymptotic theorem.', '01_introduction.tex': 'Prime gaps are often studied through their sizes, asymptotic behavior, and distributional regularities. A complementary finite-computational view is to track the residue classes of consecutive primes after removing the smallest prime divisibility constraints. For primes greater than 5, residues modulo 30 lie in\n\\[\nR=\\{1,7,11,13,17,19,23,29\\}.\n\\]\nThis gives an eight-state transition system. A first-order Markov model is a natural baseline because it records the observed probabilities of moving from one residue class to the next. However, first-order transitions need not capture memory that appears over two or more steps. This paper asks whether the empirical two-step residue transition operator contains structure beyond the first-order Markov prediction.\n\nThe central diagnostic is the residual operator\n\\[\n\\Delta=P^{(2)}-P^2,\n\\]\nwhere \\(P^{(2)}\\) is the empirical two-step transition operator and \\(P^2\\) is the two-step prediction implied by the first-order operator. If the residue process were adequately described by \\(P\\), this residual would be small and structureless up to finite-sample effects. Instead, the notebooks summarized here identify stable low-rank components in \\(\\Delta\\), validate them against synthetic controls, and package the results for reproducible inspection.', '02_methods.tex': 'Let \\(r_n\\in R\\) denote the residue class modulo 30 of the \\(n\\)-th prime greater than 5. We define the first-order transition operator\n\\[\nP_{ij}=\\Pr(r_{n+1}=j\\mid r_n=i),\n\\]\nand the empirical two-step operator\n\\[\nP^{(2)}_{ik}=\\Pr(r_{n+2}=k\\mid r_n=i).\n\\]\nThe first-order Markov baseline predicts two-step transitions as\n\\[\n(P^2)_{ik}=\\sum_j P_{ij}P_{jk}.\n\\]\nThe higher-order residual is then\n\\[\n\\Delta=P^{(2)}-P^2.\n\\]\nWe analyze \\(\\Delta\\) using the singular value decomposition\n\\[\n\\Delta=\\sum_{\\ell=1}^{8}\\sigma_\\ell u_\\ell v_\\ell^\\top.\n\\]\nLow-rank approximations retain the first \\(k\\) singular modes and compare prediction error as a function of \\(k\\). To evaluate whether the residual is a sampling artifact, we compare the real sequence to iid shuffles, Markov synthetic sequences generated from \\(P\\), block shuffles, residue-balanced shuffles, and gap-shuffle controls. Statistical validation uses null ensembles, block/window bootstrap, confidence bands, rank-stability diagnostics, and p-value summaries.', '03_results.tex': 'The empirical transition operator defines a stable finite-state baseline, but the two-step operator differs from the Markov prediction in structured ways. The residual \\(\\Delta\\) is not uniformly diffuse: heatmaps show localized positive and negative transition blocks, and the singular spectrum concentrates energy in the first few modes. In the computed range, approximately three to five modes capture most of the residual energy. Low-rank corrections reduce two-step prediction error, indicating that the leading residual modes encode reproducible transition information rather than only noise.\n\nControl comparisons strengthen this interpretation. Iid and residue-balanced shuffles substantially reduce residual strength, singular values, and mutual-information excess. Markov synthetic sequences preserve first-order statistics but do not reproduce the full two-step residual structure. Block and window-preserving resampling retain more of the structure, suggesting that the signal is related to local sequential constraints rather than marginal residue counts alone.', '04_validation.tex': 'The validation layer compares real-sequence metrics against multiple null and resampling ensembles. Metrics include two-step residual norm, Jensen--Shannon divergence, top singular value, mutual-information excess, low-rank correction improvement, and rank requirements for residual spectral energy. Structure-destroying nulls produce significant separation across several metrics, while block and window bootstrap controls are closer to the observed sequence. This is the desired validation profile: the detected structure is not explained by iid or first-order Markov models, but it is stable under controls that preserve local sequential organization.\n\nThe validation should be interpreted conservatively. These tests support a finite computational claim: within the analyzed range and representation, prime residue transitions exhibit higher-order operator structure beyond a first-order Markov baseline. They do not establish an asymptotic theorem about prime gaps or residue transitions.', '05_discussion.tex': 'The operator framework provides a compact way to study finite higher-order dependencies in prime residue sequences. Instead of directly asserting a closed-form law for prime gaps, it asks whether observed transition data contain reproducible structure after subtracting a natural first-order baseline. The low-rank residual modes suggest that part of the two-step transition behavior can be summarized by a small number of coherent components. This makes the method useful as an exploratory and validation-oriented computational tool.\n\nThe controls are especially important. Marginal residue preservation alone is insufficient to recover the real residual structure. First-order Markov simulation captures local transition probabilities but still fails to match key higher-order metrics. Block-preserving controls are closer, indicating that finite local organization matters. This supports the interpretation of \\(\\Delta\\) as a measurable higher-order memory signal rather than a simple artifact of residue frequencies.', '06_limitations.tex': 'This study is finite and computational. Results depend on the prime range, windowing choices, binning, residue abstraction, and null-model definitions. The analysis does not prove an asymptotic law and does not claim to resolve any major conjecture in analytic number theory. The empirical operators are estimates from finite data, so all claims should be read as reproducible computational evidence subject to further scaling tests and independent verification.', '07_conclusion.tex': 'Prime gap residue transitions modulo 30 exhibit measurable higher-order operator structure beyond first-order Markov baselines in finite computational ranges. The residual operator \\(\\Delta=P^{(2)}-P^2\\) contains low-rank modes, improves prediction under low-rank correction, and separates from iid and Markov null controls under statistical validation. These results motivate further study of residue-transition operators as empirical tools for detecting finite higher-order structure in prime sequences.'}
section_rows=[]
for fname, txt in sections.items():
    path = write_text(SECTIONS_DIR / fname, txt)
    section_rows.append({"section_file": f"sections/{fname}", "bytes": path.stat().st_size})
section_registry = pd.DataFrame(section_rows)
section_registry.to_csv(TABLES_DIR / "paper_section_registry.csv", index=False)
section_registry


## 5. Build figures.tex

In [ ]:
figure_blocks=[]
for _, row in fig_registry.iterrows():
    label = row["figure_id"].replace(":", "")
    block = f"""
\\begin{{figure}}[htbp]
\\centering
\\includegraphics[width=0.88\\linewidth]{{{row['paper_path']}}}
\\caption{{{row['caption']}}}
\\label{{{label}}}
\\end{{figure}}
"""
    figure_blocks.append(block.strip())
figures_tex = "\n\n".join(figure_blocks) if figure_blocks else "% No figures discovered. Run prior notebooks or upload figures."
write_text(PAPER_DIR / "figures.tex", figures_tex)
print((PAPER_DIR / "figures.tex").read_text()[:1000])


## 6. Build main.tex, references.bib, and Makefile

In [ ]:
main_tex = f"""
\\documentclass[11pt]{{article}}
\\usepackage[margin=1in]{{geometry}}
\\usepackage{{amsmath, amssymb}}
\\usepackage{{graphicx}}
\\usepackage{{booktabs}}
\\usepackage{{hyperref}}
\\usepackage{{caption}}
\\usepackage{{float}}
\\usepackage{{natbib}}
\\title{{{NOTEBOOK_TITLE}}}
\\author{{{AUTHOR}\\\\\\small \\url{{https://{REPO_URL}}}}}
\\date{{\\today}}
\\begin{{document}}
\\maketitle
\\begin{{abstract}}
\\input{{sections/00_abstract}}
\\end{{abstract}}
\\section{{Introduction}}
\\input{{sections/01_introduction}}
\\section{{Methods}}
\\input{{sections/02_methods}}
\\section{{Results}}
\\input{{sections/03_results}}
\\input{{figures}}
\\section{{Statistical Validation}}
\\input{{sections/04_validation}}
\\section{{Discussion}}
\\input{{sections/05_discussion}}
\\section{{Limitations}}
\\input{{sections/06_limitations}}
\\section{{Conclusion}}
\\input{{sections/07_conclusion}}
\\bibliographystyle{{plainnat}}
\\bibliography{{references}}
\\end{{document}}
"""
write_text(PAPER_DIR / "main.tex", main_tex)
references_bib = """
@misc{primeNumbersLab,
  author       = {Dan Hawkley},
  title        = {prime-numbers-lab: computational notebooks for prime residue transitions},
  howpublished = {GitHub repository},
  note         = {https://github.com/thinkthoughts/prime-numbers-lab}
}
@book{hardyWright,
  author    = {G. H. Hardy and E. M. Wright},
  title     = {An Introduction to the Theory of Numbers},
  publisher = {Oxford University Press},
  year      = {2008}
}
@book{coverThomas,
  author    = {Thomas M. Cover and Joy A. Thomas},
  title     = {Elements of Information Theory},
  publisher = {Wiley},
  year      = {2006}
}
"""
write_text(PAPER_DIR / "references.bib", references_bib)
makefile = """
PDF=main.pdf
TEX=main.tex
all:
\tlatexmk -pdf $(TEX)
pdflatex:
\tpdflatex $(TEX)
\tbibtex main || true
\tpdflatex $(TEX)
\tpdflatex $(TEX)
clean:
\tlatexmk -C
\trm -f *.aux *.bbl *.blg *.log *.out *.toc *.fls *.fdb_latexmk
"""
write_text(PAPER_DIR / "Makefile", makefile)
print("Wrote:", PAPER_DIR / "main.tex")


## 7. Summary and manifest

In [ ]:
summary_rows = [
    {"item": "main_claim", "value": "Prime gap residue transitions exhibit measurable higher-order operator structure beyond first-order Markov baselines in finite computational ranges."},
    {"item": "method", "value": "Compare empirical two-step operator P^(2) with Markov baseline P^2 using Delta = P^(2)-P^2."},
    {"item": "validation", "value": "Use iid, Markov synthetic, block, balanced, gap-shuffle, bootstrap, and window controls."},
    {"item": "guardrail", "value": "Empirical computational evidence only; no asymptotic theorem or prime conjecture proof claim."},
    {"item": "repo", "value": REPO_URL},
]
pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
def rebuild_manifest():
    rows=[]
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file() and path != MANIFEST_CSV:
            rows.append({"notebook_id": NOTEBOOK_ID, "relative_path": str(path.relative_to(OUTPUT_DIR)), "bytes": path.stat().st_size, "kind": path.suffix.lower().lstrip(".") or "file"})
    df=pd.DataFrame(rows); df.to_csv(MANIFEST_CSV, index=False); return df
manifest_df = rebuild_manifest()
print("Summary:", SUMMARY_CSV)
print("Manifest:", MANIFEST_CSV)
print("Files in manifest:", len(manifest_df))
manifest_df.head(20)


## 8. Optional compile check

In [ ]:
import shutil as _shutil
compiled = False
latexmk = _shutil.which("latexmk")
pdflatex = _shutil.which("pdflatex")
try:
    if latexmk:
        proc = subprocess.run([latexmk, "-pdf", "main.tex"], cwd=PAPER_DIR, capture_output=True, text=True, timeout=120)
        compiled = proc.returncode == 0
        print("latexmk return code:", proc.returncode)
        if not compiled: print(proc.stdout[-1000:], proc.stderr[-1000:])
    elif pdflatex:
        proc = subprocess.run([pdflatex, "main.tex"], cwd=PAPER_DIR, capture_output=True, text=True, timeout=120)
        compiled = proc.returncode == 0
        print("pdflatex return code:", proc.returncode)
        if not compiled: print(proc.stdout[-1000:], proc.stderr[-1000:])
    else:
        print("No LaTeX engine found; compile skipped.")
except Exception as e:
    print("Compile check skipped/failed:", repr(e))
print("compiled:", compiled)


## 9. Zip export

In [ ]:
manifest_df = rebuild_manifest()
if OUTPUT_ZIP.exists(): OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file(): z.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))
print("Created zip:", OUTPUT_ZIP)
print("Zip size bytes:", OUTPUT_ZIP.stat().st_size)
print("Output folder:", OUTPUT_DIR)


## 10. Download outputs zip (Colab standard)

In [ ]:
# Optional: download outputs bundle (template standard)
try:
    from google.colab import files
    files.download(f"{NOTEBOOK_ID}_outputs.zip")
except Exception as e:
    print("Download skipped outside Colab or file unavailable:", e)
    print("Zip path:", OUTPUT_ZIP)
